# Cross-domain adaptation benchmark — Kaggle driver

Loops over every `(dataset × representation × model × da_method × seed)` in `configs.yaml` and runs each via `run_one_config.py`.  Each run appends one row to `results.csv` under `OUTPUT_ROOT`, so the notebook can be interrupted and resumed without losing progress.

**Settings to review before running**
* `PRIORITY` — run only priorities ≤ this value (1 = stat baselines, 2 = +deep on cwru/chbmit/mimii, 3 = full matrix incl. legacy datasets).
* `DATASETS_FILTER` — empty list means "all datasets in scope"; pass e.g. `['cwru']` or `['chbmit', 'mimii']` to run a single dataset or a subset without editing `configs.yaml`.
* `SMOKE_TEST` — set to `True` to run one Priority-1 config per dataset with seed=0, then stop.  Use this once to validate that all paths resolve before launching a multi-hour job.
* `KAGGLE_*_DIR` — point these at the actual `/kaggle/input/<slug>` directories of the attached Kaggle datasets.  Leave unused vars unchanged; only the datasets referenced by your chosen priority + filter need to resolve.

In [ ]:
# ── 1. Environment ──────────────────────────────────────────────────────────
import os, sys, subprocess, json, shutil
from pathlib import Path

REPO_DIR = Path('/kaggle/working/cross-domain-experiment')
if not REPO_DIR.exists():
    raise SystemExit(f"Place the repo at {REPO_DIR} before running this notebook.")

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR))

os.environ.setdefault('DATA_ROOT',   '/kaggle/input')
os.environ.setdefault('OUTPUT_ROOT', '/kaggle/working')

# ── Dataset paths (Kaggle slugs) ─────────────────────────────────────────────
# Only the slugs for datasets you actually run need to be valid.  Datasets not
# referenced by your PRIORITY × DATASETS_FILTER selection are never touched.
#
# MIT-BIH (mondejar/mitbih-database):
#   Point at the directory that contains 100.csv, 100annotations.txt, etc.
#   Do NOT include a trailing /mitbih_database subdirectory — the per-record
#   CSVs live at the top level of the slug.
os.environ['KAGGLE_MITBIH_DIR']     = '/kaggle/input/datasets/mondejar/mitbih-database'
os.environ['KAGGLE_SLEEP_EDF_DIR']  = '/kaggle/input/datasets/labibmh/sleepedf-78/sleep-edf-database-expanded/sleep-cassette'
os.environ['KAGGLE_CWRU_DIR']       = '/kaggle/input/datasets/esraakhaled299/cwru-data'
os.environ['KAGGLE_CHBMIT_DIR']     = '/kaggle/input/datasets/bonesclarke26/combined-siena-scalp-and-chb-mit-eeg-datasets/chb-mit'
os.environ['KAGGLE_MIMII_DIR']      = '/kaggle/input/datasets/senaca/mimii-pump-sound-dataset'
os.environ['KAGGLE_MIMII_MACHINE']  = 'pump'   # senaca's slug ships only pump

PRIORITY        = 1
DATASETS_FILTER = []         # [] = all in scope; e.g. ['cwru'] or ['chbmit', 'mimii']
SMOKE_TEST      = False
OUTPUT_CSV      = Path(os.environ['OUTPUT_ROOT']) / 'results.csv'

# ── Auto-checkpoint ───────────────────────────────────────────────────────────
# Create a private Kaggle dataset at https://www.kaggle.com/datasets/new,
# then paste the slug here as "username/dataset-name".
# Leave blank to disable (results stay in /kaggle/working until session ends).
CHECKPOINT_DATASET = ''    # e.g. 'myusername/cross-domain-results'
CHECKPOINT_EVERY   = 10    # upload a new dataset version every N runs


def _kaggle_checkpoint(csv_path: Path, dataset_slug: str, label: str = '') -> None:
    """Push results.csv to a Kaggle dataset as a new version (creates on first push)."""
    stage = Path('/tmp/ckpt_results')
    stage.mkdir(exist_ok=True)
    shutil.copy(csv_path, stage / 'results.csv')
    (stage / 'dataset-metadata.json').write_text(json.dumps({
        'id': dataset_slug,
        'licenses': [{'name': 'CC0-1.0'}],
    }))
    msg = label or 'checkpoint'
    r = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', str(stage), '-m', msg],
        capture_output=True, text=True,
    )
    if r.returncode != 0 and 'not found' in r.stderr.lower():
        r = subprocess.run(
            ['kaggle', 'datasets', 'create', '-p', str(stage)],
            capture_output=True, text=True,
        )
    if r.returncode == 0:
        print(f'  ✓ checkpoint saved → kaggle.com/datasets/{dataset_slug}')
    else:
        print(f'  ✗ checkpoint failed: {r.stderr.strip()[:200]}')


print('MITBIH    =', os.environ['KAGGLE_MITBIH_DIR'])
print('SLEEP_EDF =', os.environ['KAGGLE_SLEEP_EDF_DIR'])
print('CWRU      =', os.environ['KAGGLE_CWRU_DIR'])
print('CHBMIT    =', os.environ['KAGGLE_CHBMIT_DIR'])
print('MIMII     =', os.environ['KAGGLE_MIMII_DIR'], f"(machine={os.environ['KAGGLE_MIMII_MACHINE']})")
print('PRIORITY  =', PRIORITY)
print('FILTER    =', DATASETS_FILTER or '(all in scope)')
print('SMOKE     =', SMOKE_TEST)
print('CHECKPOINT=', CHECKPOINT_DATASET or '(disabled — set CHECKPOINT_DATASET to enable)')

In [ ]:
# ── 2. Install runtime dependencies (only those Kaggle's base image lacks) ──
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'mne', 'wfdb', 'scikit-posthocs',
])

In [ ]:
# ── 3. Expand configs.yaml into a flat list of run dicts ────────────────────
import itertools, yaml

with open(REPO_DIR / 'configs.yaml') as f:
    cfg = yaml.safe_load(f)

# Default seeds for any priority that does not override.  Priority 2 (deep)
# typically overrides with 5 seeds; Priority 1 (stat) keeps the global 3.
DEFAULT_SEEDS = cfg['seeds']

def expand(priorities_to_run):
    runs = []
    for prio in priorities_to_run:
        prio_cfg = cfg['priorities'][prio]
        prio_seeds = prio_cfg.get('seeds', DEFAULT_SEEDS)
        for group in prio_cfg['groups']:
            datasets = group['datasets']
            if DATASETS_FILTER:
                datasets = [d for d in datasets if d in DATASETS_FILTER]
            for ds, repr_, model, da in itertools.product(
                datasets,
                group['representations'],
                group['models'],
                group['da_methods'],
            ):
                for seed in prio_seeds:
                    runs.append({
                        'priority':       prio,
                        'dataset':        ds,
                        'representation': repr_,
                        'model':          model,
                        'da_method':      da,
                        'seed':           seed,
                    })
    return runs

if SMOKE_TEST:
    # One stat-source-only config per dataset in scope (priority 1 stat baseline).
    smoke_datasets = DATASETS_FILTER or ['mitbih', 'sleep-edf', 'cwru', 'chbmit', 'mimii']
    runs_all = [
        {'priority': 1, 'dataset': ds, 'representation': 'raw',
         'model': 'logreg', 'da_method': 'source-only', 'seed': 0}
        for ds in smoke_datasets
    ]
else:
    runs_all = expand([p for p in (1, 2, 3) if p <= PRIORITY])

print(f'Expanded {len(runs_all)} (config, seed) runs')

In [ ]:
# ── 4. Skip already-done runs ───────────────────────────────────────────────
import csv

done = set()
if OUTPUT_CSV.exists():
    with open(OUTPUT_CSV, newline='') as f:
        for row in csv.DictReader(f):
            done.add((row['dataset'], row['representation'], row['model'],
                      row['da_method'], int(row['seed'])))

runs_pending = [
    r for r in runs_all
    if (r['dataset'], r['representation'], r['model'], r['da_method'], r['seed']) not in done
]
print(f'{len(done)} already done, {len(runs_pending)} pending')

In [ ]:
# ── 5. Main loop ────────────────────────────────────────────────────────────
import time

N = len(runs_pending)
elapsed_per_run = []
LOG_FILE = OUTPUT_CSV.parent / 'run.log'

for i, run in enumerate(runs_pending, 1):
    cmd = [
        sys.executable, str(REPO_DIR / 'run_one_config.py'),
        '--dataset',        run['dataset'],
        '--representation', run['representation'],
        '--model',          run['model'],
        '--da-method',      run['da_method'],
        '--seed',           str(run['seed']),
        '--output-csv',     str(OUTPUT_CSV),
    ]

    # Print "starting" immediately — gives visual proof the loop is not stuck
    print(f'[{i:4d}/{N}] ▶  {run["dataset"]:10s} {run["representation"]:11s} '
          f'{run["model"]:14s} {run["da_method"]:18s} seed={run["seed"]}', flush=True)

    t0 = time.time()
    with open(LOG_FILE, 'a') as lf:
        lf.write(f'\n=== [{i}/{N}] {" ".join(str(c) for c in cmd[2:])} ===\n')
        proc = subprocess.run(cmd, stdout=lf, stderr=subprocess.STDOUT, text=True)
    dt = time.time() - t0
    elapsed_per_run.append(dt)

    # Exit code 2 = compatibility skip; anything else is a real error
    tag = 'OK ' if proc.returncode == 0 else ('SKIP' if proc.returncode == 2 else 'ERR ')
    avg = sum(elapsed_per_run) / len(elapsed_per_run)
    eta_h = avg * (N - i) / 3600
    print(f'[{i:4d}/{N}] {tag}  {run["dataset"]:10s} {run["representation"]:11s} '
          f'{run["model"]:14s} {run["da_method"]:18s} seed={run["seed"]}  '
          f'{dt:6.1f}s  ETA≈{eta_h:5.2f}h')
    if proc.returncode not in (0, 2):
        # Show tail of log for errors
        with open(LOG_FILE) as lf:
            tail = lf.readlines()[-8:]
        print('    log tail:')
        for line in tail:
            print('      ' + line.rstrip())

    # Auto-checkpoint every CHECKPOINT_EVERY completed runs
    if CHECKPOINT_DATASET and OUTPUT_CSV.exists() and i % CHECKPOINT_EVERY == 0:
        _kaggle_checkpoint(OUTPUT_CSV, CHECKPOINT_DATASET, f'run {i}/{N}')

# Final checkpoint — always save when the full loop finishes
if CHECKPOINT_DATASET and OUTPUT_CSV.exists():
    _kaggle_checkpoint(OUTPUT_CSV, CHECKPOINT_DATASET,
                       f'final – {N} runs complete')

print(f'\nDone. {N} runs processed.')

In [ ]:
# ── 5b. (OPTIONAL) Targeted neural-gap run from manifest ─────────────────────
#
# Runs ONLY the configs in neural_gap_manifest.csv — the canonical deep-DA
# matrix (raw×InceptionTime, raw×PatchTST, log-stft×ResNet2D, cwt×ResNet2D)
# × {source-only, deep-coral, codats, mk-mmd} on the datasets where neural
# methods were not yet run (mitbih, sleep-edf, cwru, + completing mimii).
#
# Independent of PRIORITY and of configs/compatibility.yaml: every pair in the
# manifest is canonical (always compatible), so the spectral×1-D-model
# crossings still under discussion are NOT touched here.
#
# Appends to the SAME results.csv and skips rows already present → restart-safe.
# Set RUN_NEURAL_GAP = True to use this instead of (or in addition to) cell 5.
RUN_NEURAL_GAP = False

if RUN_NEURAL_GAP:
    import csv, time

    with open(REPO_DIR / 'neural_gap_manifest.csv', newline='') as f:
        gap_runs = [
            {'dataset': r['dataset'], 'representation': r['representation'],
             'model': r['model'], 'da_method': r['da_method'], 'seed': int(r['seed'])}
            for r in csv.DictReader(f)
        ]
    if DATASETS_FILTER:                         # optional: one dataset at a time
        gap_runs = [r for r in gap_runs if r['dataset'] in DATASETS_FILTER]

    done_gap = set()
    if OUTPUT_CSV.exists():
        with open(OUTPUT_CSV, newline='') as f:
            for row in csv.DictReader(f):
                done_gap.add((row['dataset'], row['representation'], row['model'],
                              row['da_method'], int(row['seed'])))
    gap_pending = [
        r for r in gap_runs
        if (r['dataset'], r['representation'], r['model'], r['da_method'], r['seed']) not in done_gap
    ]
    print(f'Neural-gap: {len(gap_runs)} in manifest, {len(gap_pending)} pending')

    N = len(gap_pending)
    elapsed = []
    for i, run in enumerate(gap_pending, 1):
        cmd = [
            sys.executable, str(REPO_DIR / 'run_one_config.py'),
            '--dataset', run['dataset'], '--representation', run['representation'],
            '--model', run['model'], '--da-method', run['da_method'],
            '--seed', str(run['seed']), '--output-csv', str(OUTPUT_CSV),
        ]
        print(f'[{i:4d}/{N}] ▶  {run["dataset"]:10s} {run["representation"]:11s} '
              f'{run["model"]:14s} {run["da_method"]:12s} seed={run["seed"]}', flush=True)
        t0 = time.time()
        with open(OUTPUT_CSV.parent / 'run.log', 'a') as lf:
            lf.write(f'\n=== [gap {i}/{N}] {" ".join(str(c) for c in cmd[2:])} ===\n')
            proc = subprocess.run(cmd, stdout=lf, stderr=subprocess.STDOUT, text=True)
        dt = time.time() - t0
        elapsed.append(dt)
        tag = 'OK ' if proc.returncode == 0 else ('SKIP' if proc.returncode == 2 else 'ERR ')
        eta_h = (sum(elapsed) / len(elapsed)) * (N - i) / 3600
        print(f'[{i:4d}/{N}] {tag}  {dt:7.1f}s  ETA≈{eta_h:5.2f}h')
        if proc.returncode not in (0, 2):
            with open(OUTPUT_CSV.parent / 'run.log') as lf:
                for line in lf.readlines()[-8:]:
                    print('      ' + line.rstrip())
        if CHECKPOINT_DATASET and OUTPUT_CSV.exists() and i % CHECKPOINT_EVERY == 0:
            _kaggle_checkpoint(OUTPUT_CSV, CHECKPOINT_DATASET, f'neural-gap {i}/{N}')

    if CHECKPOINT_DATASET and OUTPUT_CSV.exists():
        _kaggle_checkpoint(OUTPUT_CSV, CHECKPOINT_DATASET, f'neural-gap final – {N} runs')
    print(f'\nNeural-gap done. {N} runs processed.')

In [ ]:
# ── 6. Aggregate + ANOVA ────────────────────────────────────────────────────
# Skip on smoke runs — they don't have enough rows for ANOVA.
if not SMOKE_TEST:
    subprocess.check_call([
        sys.executable, str(REPO_DIR / 'scripts' / 'aggregate.py'),
        '--results-dir', str(OUTPUT_CSV.parent),
        '--out-dir',     str(OUTPUT_CSV.parent),
    ])